In [2]:
import requests
import pandas as pd
import numpy as np
from gprofiler import GProfiler
from collections import defaultdict
from typing import List, Dict
import spacy
import xml.etree.ElementTree as ET
import re
nlp = spacy.load("en_core_web_sm")

In [3]:
API_URL = "https://api.platform.opentargets.org/api/v4/graphql"
DISEASE_ID = "MONDO_0005180"
query = f"""
query {{
  disease(efoId: "{DISEASE_ID}") {{
    associatedTargets(page: {{ index: 0, size: 100 }}) {{
      count
      rows {{
        target {{
          id
          approvedSymbol
          approvedName
        }}
        score
      }}
    }}
  }}
}}
"""
response = requests.post(API_URL, json={"query":query})
if response.status_code != 200:
    raise Exception("Something went wrong while fetching genes")
else:
    target_genes_response = response.json()
print(target_genes_response)



{'data': {'disease': {'associatedTargets': {'count': 6584, 'rows': [{'target': {'id': 'ENSG00000188906', 'approvedSymbol': 'LRRK2', 'approvedName': 'leucine rich repeat kinase 2'}, 'score': 0.8789207866590859}, {'target': {'id': 'ENSG00000145335', 'approvedSymbol': 'SNCA', 'approvedName': 'synuclein alpha'}, 'score': 0.8585012729359579}, {'target': {'id': 'ENSG00000159363', 'approvedSymbol': 'ATP13A2', 'approvedName': 'ATPase cation transporting 13A2'}, 'score': 0.8525516685262865}, {'target': {'id': 'ENSG00000185345', 'approvedSymbol': 'PRKN', 'approvedName': 'parkin RBR E3 ubiquitin protein ligase'}, 'score': 0.8505122712454569}, {'target': {'id': 'ENSG00000158828', 'approvedSymbol': 'PINK1', 'approvedName': 'PTEN induced kinase 1'}, 'score': 0.8489915009692078}, {'target': {'id': 'ENSG00000116675', 'approvedSymbol': 'DNAJC6', 'approvedName': 'DnaJ heat shock protein family (Hsp40) member C6'}, 'score': 0.813216808888733}, {'target': {'id': 'ENSG00000116288', 'approvedSymbol': 'PARK7

In [4]:
def get_target_genes(response:dict) -> dict:
    target_res = response['data']['disease']['associatedTargets']['rows']
    genes = []
    for i in target_res:
        gene_data = {
            "gene_id": i['target']['id'],
            "symbol":  i['target']['approvedSymbol'],
            "name": i['target']['approvedName'],
            "score": i['score']
        }
        genes.append(gene_data)
    return genes

genes_target = get_target_genes(target_genes_response)
print(genes_target)

[{'gene_id': 'ENSG00000188906', 'symbol': 'LRRK2', 'name': 'leucine rich repeat kinase 2', 'score': 0.8789207866590859}, {'gene_id': 'ENSG00000145335', 'symbol': 'SNCA', 'name': 'synuclein alpha', 'score': 0.8585012729359579}, {'gene_id': 'ENSG00000159363', 'symbol': 'ATP13A2', 'name': 'ATPase cation transporting 13A2', 'score': 0.8525516685262865}, {'gene_id': 'ENSG00000185345', 'symbol': 'PRKN', 'name': 'parkin RBR E3 ubiquitin protein ligase', 'score': 0.8505122712454569}, {'gene_id': 'ENSG00000158828', 'symbol': 'PINK1', 'name': 'PTEN induced kinase 1', 'score': 0.8489915009692078}, {'gene_id': 'ENSG00000116675', 'symbol': 'DNAJC6', 'name': 'DnaJ heat shock protein family (Hsp40) member C6', 'score': 0.813216808888733}, {'gene_id': 'ENSG00000116288', 'symbol': 'PARK7', 'name': 'Parkinsonism associated deglycase', 'score': 0.8106296934517689}, {'gene_id': 'ENSG00000100225', 'symbol': 'FBXO7', 'name': 'F-box protein 7', 'score': 0.8055069071068177}, {'gene_id': 'ENSG00000177628', 'sy

In [5]:
genes_target_prelim_df = pd.DataFrame(genes_target)
genes_target_prelim_df

,gene_id,symbol,name,score
0,ENSG00000188906,LRRK2,leucine rich repeat kinase 2,0.878921
1,ENSG00000145335,SNCA,synuclein alpha,0.858501
2,ENSG00000159363,ATP13A2,ATPase cation transporting 13A2,0.852552
3,ENSG00000185345,PRKN,parkin RBR E3 ubiquitin protein ligase,0.850512
4,ENSG00000158828,PINK1,PTEN induced kinase 1,0.848992
...,...,...,...,...
95,ENSG00000010810,FYN,"FYN proto-oncogene, Src family tyrosine kinase",0.497532
96,ENSG00000113712,CSNK1A1,casein kinase 1 alpha 1,0.497218
97,ENSG00000169676,DRD5,dopamine receptor D5,0.496620
98,ENSG00000164615,CAMLG,calcium modulating ligand,0.494202


In [6]:
def load_gtex_expression(filepath: str):
    df = pd.read_csv(filepath, sep="\t", skiprows=2, compression='gzip')
    brain_cols = [col for col in df.columns if 'Brain' in col]
    df["median_tpm_brain"] = df[brain_cols].median(axis=1)
    df = df.rename(columns={"Name": "gene_id", "Description": "symbol"})
    df["gene_id_stripped"] = df["gene_id"].str.split(".").str[0]
    return df[["gene_id_stripped", "symbol", "median_tpm_brain"]]

gene_tpm_brain_df = load_gtex_expression("GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm (1).gct.gz")

def merge_open_targets_gtex(df_1, df_2):
    merged_df = df_1.merge(
        df_2,
        left_on="gene_id",
        right_on="gene_id_stripped",
        how="left"
    )
    merged_df = merged_df.drop(columns=["gene_id_stripped"])
    return merged_df

mereged_genes_parkinsons = merge_open_targets_gtex(genes_target_prelim_df, gene_tpm_brain_df)


In [7]:
mereged_genes_parkinsons
def prioritize_genes(df):
    new_df = df.copy()
    new_df["median_tpm_brain"] = new_df["median_tpm_brain"].fillna(0)
    new_df["final_score"] =  np.log2(1+new_df["median_tpm_brain"]) * new_df["score"]
    new_df = new_df.sort_values(by="final_score", ascending=False)
    return new_df

final_genes_parkinsons_sorted = prioritize_genes(mereged_genes_parkinsons)


In [8]:
final_genes_parkinsons_sorted

,gene_id,symbol_x,name,score,symbol_y,median_tpm_brain,final_score
6,ENSG00000116288,PARK7,Parkinsonism associated deglycase,0.810630,PARK7,182.992000,6.098772
15,ENSG00000197746,PSAP,prosaposin,0.644604,PSAP,633.102000,6.000345
2,ENSG00000159363,ATP13A2,ATPase cation transporting 13A2,0.852552,ATP13A2,59.661500,5.049416
36,ENSG00000106153,CHCHD2,coiled-coil-helix-coiled-coil-helix domain con...,0.586215,CHCHD2,322.936000,4.888780
47,ENSG00000168653,NDUFS5,NADH:ubiquinone oxidoreductase subunit S5,0.544431,NDUFS5,428.624000,4.762104
...,...,...,...,...,...,...,...
59,ENSG00000178999,AURKB,aurora kinase B,0.526138,AURKB,0.053693,0.039700
18,ENSG00000151577,DRD3,dopamine receptor D3,0.629745,DRD3,0.011972,0.010812
99,ENSG00000094755,GABRP,gamma-aminobutyric acid type A receptor subuni...,0.491403,GABRP,0.015352,0.010801
88,ENSG00000103546,SLC6A2,solute carrier family 6 member 2,0.499819,SLC6A2,0.007738,0.005558


In [9]:
def run_gprofiler(symbols: list):
    gp = GProfiler(return_dataframe=True)
    results = gp.profile(organism='hsapiens', query=symbols, no_evidences=False)
    return results

gprofiler_res = run_gprofiler(final_genes_parkinsons_sorted.head(50)['symbol_x'].dropna().tolist())

In [10]:
gprofiler_res.to_csv('OUTPUT_CSV.csv', index=False)

In [11]:
def enrich_top_200_genes(gprofiler_top):
    enriched_score = defaultdict(float)
    for _, row in gprofiler_top.iterrows():
        if (len(row["intersections"]) == 0):
            continue
        for i in row["intersections"]:
            enriched_score[i] += -np.log10(row["p_value"])/len(row["intersections"])
    return enriched_score

enriched_score = enrich_top_200_genes(gprofiler_res)


In [12]:
enriched_score

defaultdict(float,
            {'PARK7': np.float64(121.5375532654405),
             'PSAP': np.float64(31.40551603777035),
             'ATP13A2': np.float64(65.1683109253922),
             'CHCHD2': np.float64(36.74917827530963),
             'SNCA': np.float64(126.88162502179861),
             'UCHL1': np.float64(65.59297273493974),
             'FBXO7': np.float64(42.943653448993686),
             'MAPT': np.float64(65.73791684532166),
             'EIF4G1': np.float64(54.17538887267741),
             'VPS35': np.float64(83.74497758722529),
             'UQCRC1': np.float64(50.137011537967325),
             'PTPA': np.float64(20.501792481830638),
             'GBA1': np.float64(72.90899859908062),
             'PLA2G6': np.float64(43.07518869425227),
             'HTRA2': np.float64(57.31597835436296),
             'PINK1': np.float64(118.9841456419906),
             'DNAJC6': np.float64(57.304407678769785),
             'COMT': np.float64(30.716059337324523),
             'ATP6V1A

In [13]:
def compute_enriched_score(df_200, g_profiler_scores_dict, alpha):
    df_200_final = df_200.copy()
    df_200_final["g_profiler_scores"] = df_200_final["symbol_x"].map(g_profiler_scores_dict).fillna(0)
    df_200_final["final_gene_score"] = df_200_final["final_score"] + alpha*df_200_final["g_profiler_scores"]
    return df_200_final.sort_values(by="final_gene_score", ascending=False)

final_genes_scored = compute_enriched_score(final_genes_parkinsons_sorted.head(50), enriched_score, 1.0)

In [14]:
final_genes_scored.head(5)

,gene_id,symbol_x,name,score,symbol_y,median_tpm_brain,final_score,g_profiler_scores,final_gene_score
1,ENSG00000145335,SNCA,synuclein alpha,0.858501,SNCA,38.2467,4.545335,126.881625,131.426960
6,ENSG00000116288,PARK7,Parkinsonism associated deglycase,0.810630,PARK7,182.9920,6.098772,121.537553,127.636325
4,ENSG00000158828,PINK1,PTEN induced kinase 1,0.848992,PINK1,41.0024,4.578102,118.984146,123.562247
9,ENSG00000069329,VPS35,VPS35 retromer complex component,0.760328,VPS35,19.3967,3.307626,83.744978,87.052604
47,ENSG00000168653,NDUFS5,NADH:ubiquinone oxidoreductase subunit S5,0.544431,NDUFS5,428.6240,4.762104,80.410814,85.172917


In [15]:
# check literature hits for the genes shortlisted in Workflow 1

def check_eu_pmc(disease: str, gene: str) -> dict:
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    try:
        response = requests.get(url, {
            "query": f'"{gene}" AND "{disease}"',
            "format": "json",
            "pageSize": 100
        })
        response.raise_for_status()
        data = response.json()
        hit_count = int(data.get("hitCount", 0))
        res = data.get("resultList", {}).get("result", [])
        years = [int(x["pubYear"]) for x in res if "pubYear" in x]
        fulltext_hits = sum(1 for paper in res if paper.get("hasTextMinedTerms") == "Y")
        return {
            "gene":gene,
            "hit_count":hit_count,
            "fulltext_hits":fulltext_hits,
            "mean_pub_year": np.mean(years) if years else np.nan
        }
    except Exception as e:
        raise RuntimeError(f"Error while fetching from EU PMC: {str(e)}")


In [16]:
def check_literature_match_for_genes(gene_list: list, disease: str) -> dict:
    return [check_eu_pmc(disease, x) for x in gene_list]

genes_lit_match = check_literature_match_for_genes(final_genes_scored["symbol_x"].to_list(), "Parkinson's Disease")
genes_lit_match_df = pd.DataFrame(genes_lit_match)

In [17]:
genes_lit_match_df.head(5)

,gene,hit_count,fulltext_hits,mean_pub_year
0,SNCA,7608,97,2024.86
1,PARK7,2316,98,2024.44
2,PINK1,6453,98,2024.75
3,VPS35,1397,100,2024.27
4,NDUFS5,50,50,2018.24


In [21]:
class LiteratureFetcher:

    def __init__(self, gene_list: List[str], disease_name: str, page_size: int = 20):
        self.gene_list = gene_list
        self.disease_name = disease_name
        self.page_size = page_size
        self._base_url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

    def fetch_articles(self) -> List[Dict]:
        gene_wise_articles = []
        for gene in self.gene_list:
            query = f"{gene} AND {self.disease_name}"
            try:
                response = requests.get(self._base_url, params={
                    "query": query,
                    "format": "json",
                    "pageSize": self.page_size
                })
            except Exception as e:
                print(f"Error for gene: {gene} -> {e}")
                continue
            articles = []
            for i in response.json().get("resultList", {}).get("result", []):
                if "pmcid" in i:
                    articles.append({
                        "pmcid": i["pmcid"],
                        "pmid": i.get("pmid", ""),
                        "title": i.get("title", ""),
                        "source": i.get("source", ""),
                        "journal": i.get("journalTitle", "")
                    })
            gene_wise_articles.append({"gene":gene, "articles":articles})
        return gene_wise_articles


fetcher = LiteratureFetcher(final_genes_scored["symbol_x"].to_list(), "Parkinson's Disease")
article_list = fetcher.fetch_articles()